# 03 — Black–Litterman

`BlackLitterman` (in `maplab.models`), a long-only tangency portfolio on a Black–Litterman posterior mean (equal-weight prior, 12−1 momentum sign views), scored by the rolling backtest against EW and 60/40.

## §1 Black–Litterman and the idea

Black & Litterman (1992) replace an estimated μ with a prior implied by a reference portfolio, the returns that make it mean-variance optimal, and blend in explicit views with stated confidence. He & Litterman (1999) set each view's uncertainty proportional to the prior's; this notebook does the same. Notebook 01 showed a sample-mean μ turns the optimizer into an error maximizer, and notebook 02 that a CAPM μ collapses it to 100% SPY. Here the prior is implied by EW, so with no views the model holds EW. The views are signs of trailing excess returns, as in time-series momentum (Moskowitz, Ooi & Pedersen 2012, 12 months); here the latest month is skipped.

## §2 The model: prior, views, posterior

**General form.** Returns have an uncertain mean, $\mu \sim N(\Pi, \tau\Sigma)$, and views state $Q = P\mu + \varepsilon$ with $\varepsilon \sim N(0, \Omega)$. The posterior mean (Black & Litterman 1992) is

$$\mu_{BL} = \left[(\tau\Sigma)^{-1} + P'\Omega^{-1}P\right]^{-1}\left[(\tau\Sigma)^{-1}\Pi + P'\Omega^{-1}Q\right] = \Pi + \tau\Sigma P'(P\tau\Sigma P' + \Omega)^{-1}(Q - P\Pi),$$

with posterior covariance of the mean $M = \left[(\tau\Sigma)^{-1} + P'\Omega^{-1}P\right]^{-1}$. Precise views (small $\Omega$) pull $\mu_{BL}$ toward $Q$; vague views leave it at $\Pi$.

**Prior.** $\Pi = \delta\,\Sigma\,w_{EW}$ with $\delta = \mathrm{sr}_{ref}/\sigma_{EW}$, so EW's implied Sharpe ratio is $\mathrm{sr}_{ref} = 0.3$, fixed a priori. The long-only tangency on $\Pi$ is $w_{EW}$ exactly.

**Views.** One absolute view per asset ($P = I$): $Q = \Pi + k\, s \circ \sigma$, where $s_i$ is the sign of asset $i$'s trailing 12−1 excess return and $\sigma_i$ its volatility. Each view moves an asset's Sharpe ratio $k$ above or below the prior's.

**Confidence.** $\Omega = \tau D$ with $D = \mathrm{diag}(\Sigma)$ (He & Litterman 1999): each view is as uncertain as the prior on that asset. Then

$$\mu_{BL} = \Pi + k\,\Sigma(\Sigma + D)^{-1}(s \circ \sigma), \qquad \Sigma^{-1}\mu_{BL} = \delta\, w_{EW} + k\,(\Sigma + D)^{-1}(s \circ \sigma).$$

$\tau$ cancels, and $k = 0$ returns EW. The unconstrained tangency is EW plus a momentum tilt, sized relative to EW by $k\,\sigma_{EW}/\mathrm{sr}_{ref}$. If $\Sigma$ were diagonal, $\mu_{BL}$ would sit halfway between $\Pi$ and $Q$.

**The optimization problem.** The unconstrained tangency shorts and, as k grows, levers up (68.9× gross at k = 0.4, §4.4), so the class solves the long-only version on the same 252-day $\Sigma$:

$$\max_w \; \frac{w'\mu_{BL}}{\sqrt{w'\Sigma w}} \quad \text{s.t.} \quad \mathbf{1}'w = 1, \; w \ge 0 .$$

$\mu_{BL}$ is already an excess return, so no $r_f$ enters. Weights use $\Sigma$, not $\Sigma + M$ with $M = \tau\Sigma(\Sigma + D)^{-1}D$; $M$ scales with $\tau$, so using $\Sigma$ keeps $\tau$ out of the weights.

**k.** $k = 0.2$ was planned. A snapshot check at three train-window dates showed $k \ge 0.2$ pushes effective N below 5 at 2022-12-31, so $k = 0.1$ was fixed before the backtest and $0.2$ kept as a sensitivity (§4.4).

**Inputs.** $\Sigma$ and momentum come from the same trailing 252 daily log returns before the rebalance date; momentum sums the first 231 of them (days $t-252$ to $t-22$) in excess of BIL via $\log(1 + r_f)$.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import maplab as ml
import _nb03_helpers as h
from maplab import Panel, apply_style, BlackLitterman, EqualWeight, FixedWeight, backtest, summary, tangency_closed_form

apply_style()
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # estimation (Sigma, momentum)
simple_returns = ml.to_simple_returns(prices)  # compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

print(f"prices: {prices.shape[0]:,} days × {prices.shape[1]} assets "
      f"({prices.index.min().date()} → {prices.index.max().date()})")

## §3 The class and the rolling backtest

`BlackLitterman` takes Σ from the shared `Strategy._estimate` (printed in notebook 01 §3). `posterior`, printed below, builds Π, the views and μ_BL; `predict_weights` solves the §2 problem on μ_BL (Solver below).

In [ ]:
import inspect
import textwrap

fn = BlackLitterman.posterior
print(f"# ── {fn.__qualname__} " + "─" * 40)
print(textwrap.dedent(inspect.getsource(fn)))

**Solver.** `predict_weights` runs the long-only tangency QP shared with MaxSharpe (notebook 01 §3), SLSQP with analytic gradients. If no asset has $\mu_{BL} > 0$ the class falls back to GMV and logs the date; a failed solve is retried and logged. Rebalances with effective N below 5 are logged as concentrated; the weights are not changed.

**Backtest** as in notebook 01 §3.

## §4 Inputs and snapshot

One rebalance, 2022-12-31: the prior, the views, the posterior and the weights.

### §4.1 Why not a CAPM prior

A prior implied by 100% SPY, $\Pi = \delta\,\Sigma\,e_{SPY}$, gives $\Sigma^{-1}\Pi \propto e_{SPY}$: the no-view portfolio is 100% SPY, the collapse of notebook 02 §4.2.

In [ ]:
rebal_le_split = ml.rebalance_dates(log_returns.index)
asof_split = rebal_le_split[rebal_le_split <= split_ts].max()
print(f"snapshot asof (last rebalance <= split): {asof_split.date()}")

bl_probe = BlackLitterman()
post_probe = bl_probe.posterior(panel, asof_split)
Sigma_probe = post_probe["Sigma"]
delta_probe = post_probe["delta"]

e_spy = pd.Series(0.0, index=ml.UNIVERSE)
e_spy["SPY"] = 1.0
Pi_capm = delta_probe * (Sigma_probe.to_numpy() @ e_spy.to_numpy())
Pi_capm = pd.Series(Pi_capm, index=ml.UNIVERSE)

w_capm_tangency = tangency_closed_form(Pi_capm, Sigma_probe, 0.0)
print(f"CAPM-implied prior -> unconstrained tangency SPY weight: {w_capm_tangency['SPY']:.6f}")
print(f"max |other assets|: {w_capm_tangency.drop('SPY').abs().max():.2e}")

**Reading.** The CAPM prior's tangency is 100% SPY, every other weight at most 2.04e-15. With that prior the views would be read against SPY instead of the EW benchmark.

### §4.2 Posterior at the snapshot

Π, σ, 12−1 excess momentum, its sign, Q and μ_BL at k = 0.1 (annualized, %). Then the §2 checks: k = 0 returns EW, τ cancels, the closed form holds. Last, the unconstrained tangency on μ_BL: how hard the views pull before the long-only constraint.

In [ ]:
bl01 = BlackLitterman(k=0.1)
post = bl01.posterior(panel, asof_split)

mech_table = h.mechanics_table(post)
print(f"delta = {post['delta']:.4f}   #Pi<0 = {int((post['pi'] < 0).sum())}")
mech_table.round(3)

In [ ]:
chk = h.design_checks(panel, asof_split, post)

# check 1: k=0 -> long-only weights == EW, mu_bl == pi
print(f"[k=0 -> EW] max|w_k0 - w_EW| = {chk['k0_dev']:.2e}")

# check 2: tau invariance
print(f"[tau invariance] max|mu_BL(tau=0.01) - mu_BL(tau=1.0)| = "
      f"{chk['tau_dev']:.2e}")

# check 3: closed-form identity
print(f"[closed-form identity] max|LHS - RHS| = {chk['identity_dev']:.2e}")

# unconstrained BL tangency
w_unc = chk["w_unc"]
top_long = w_unc.sort_values(ascending=False).head(3)
negatives = w_unc[w_unc < 0].sort_values()
top_short = negatives.head(3)
print(f"\n[unconstrained BL(k=0.1) tangency]  gross leverage = {w_unc.abs().sum():.3f}")
print(f"  largest longs:  {dict(top_long.round(3))}")
if len(top_short):
    print(f"  largest shorts: {dict(top_short.round(3))}")
else:
    print("  largest shorts: none")

**Reading.** δ = 2.7181. Only UUP has a negative prior (Π = −1.80%): Π_i is proportional to cov(r_i, r_EW), and UUP moves against the EW book. Eleven of 13 momentum signs are negative; only DBC and UUP are positive. The checks hold: k = 0 gives EW to 3.66e-07 (the SLSQP tolerance), τ cancels to 6.94e-18, the identity holds to 2.97e-15, and the unconstrained tangency runs at 1.131 gross.

### §4.3 Prior vs posterior

Π and μ_BL for k = 0.1 and 0.2 at 2022-12-31.

In [ ]:
fig = h.prior_posterior_figure(panel, asof_split)
plt.show()

**Reading.** The views pull the eleven negative-momentum assets down and DBC and UUP up. UUP's posterior stays negative (−1.03%), yet BL(0.1) holds 19.8% of it (§4.4): as with GLD in notebook 01, a held asset with a negative expected return must hedge the rest of the portfolio. With no cash asset, a long-only book can express this risk-off tilt only through DBC and UUP.

### §4.4 k as a concentration knob

Effective N, largest weight and unconstrained gross leverage for k = 0.1, 0.2 and 0.4, then the long-only weights against EW. k = 0.4 is snapshot only, never backtested.

In [ ]:
k_table, wtab = h.k_sensitivity(panel, asof_split)

print(k_table.round(3))
print()
print("long-only weights (%):")
print((100 * wtab).round(1))

In [ ]:
fig = h.ew_vs_bl_weights(wtab)
plt.show()

**Reading.** Effective N falls 7.58 → 3.47 → 1.78 and unconstrained gross leverage rises 1.131 → 2.038 → 68.899. Eleven negative signs are one risk-off bet, not 13 signals, and a fully invested long-only book can place it only in DBC and UUP.

### §4.5 Capital vs risk

Capital weights (dark) and shares of portfolio variance (light) at 2022-12-31; a negative share is a hedge. Table columns as in notebook 01 §4.5.

In [ ]:
SNAP = asof_split
Sigma_2022 = post["Sigma"]
w_2022 = {
    "BL(0.1)": wtab["k=0.1"],
    "BL(0.2)": wtab["k=0.2"],
    "EqualWeight": EqualWeight()(panel, SNAP),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40")(panel, SNAP),
}
w_2022 = {name: w.reindex(Sigma_2022.columns) for name, w in w_2022.items()}

weights_c5 = {"BL(0.1)": w_2022["BL(0.1)"], "BL(0.2)": w_2022["BL(0.2)"],
              "EW": w_2022["EqualWeight"], "60/40": w_2022["60/40"]}
print(ml.diagnostics.diversification_table(weights_c5, Sigma_2022).to_string(float_format=lambda v: f"{v:.4f}"))

fig = ml.plotting.capital_vs_risk(weights_c5, Sigma_2022, title="Capital weight vs risk share, 2022-12-31 (sample Σ)")
plt.show()

**Reading.** BL(0.1) holds 19.8% UUP and spreads over 2.18 uncorrelated bets, BL(0.2) 41.8% and 3.28 (EW 1.40, 60/40 1.76). The largest single variance share is 38% for BL(0.1) and 84% for BL(0.2).

## §5 Out-of-sample results

Walk-forward as in notebook 01 §5; design choices, k included, used data to 2022, so the test window is the cleaner read. BL(0.1) is the model, BL(0.2) its sensitivity; EW is BL with k = 0. The cell ends with the reproduction gate.

In [ ]:
strategies = {
    "BL(0.1)": BlackLitterman(k=0.1),
    "BL(0.2)": BlackLitterman(k=0.2),
    "EqualWeight": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

# §4 snapshot weights are the backtest's weights at 2022-12-31
for name, w in w_2022.items():
    gap = float((results[name]["wlog"].loc[SNAP].reindex(w.index) - w).abs().max())
    assert gap <= 1e-12, (name, gap)

rows = []
for name, r in results.items():
    net = r["net"]
    turnover = r["diag"]["turnover_per_rebalance"]
    win_defs = {
        "full": (net.index >= net.index.min(), turnover.index >= turnover.index.min()),
        "train (<= split)": (net.index <= split_ts, turnover.index <= split_ts),
        "test (> split)": (net.index > split_ts, turnover.index > split_ts),
    }
    for window_name, (net_sel, t_sel) in win_defs.items():
        s = summary(net.loc[net_sel], turnover.loc[t_sel], rf=rf_daily)
        s["strategy"] = name
        s["window"] = window_name
        rows.append(s)

summary_table = pd.DataFrame(rows).set_index(["strategy", "window"])
summary_table = summary_table[["ann_return", "ann_vol", "sharpe", "max_dd", "calmar", "hit_rate", "ann_turnover"]]

for name in ("BL(0.1)", "BL(0.2)"):
    s = results[name]["strat"]
    print(f"{name} fallback/retry/concentrated (effN < 5): {len(s.fallback_dates)}/{len(s.retry_dates)}/"
          f"{len(s.concentration_dates)} of {results[name]['diag']['n_rebalances']}")
print("BL(0.1) concentrated dates:", ", ".join(str(d.date()) for d in results["BL(0.1)"]["strat"].concentration_dates))

ml.inference.check_reproduction(summary_table, {
    "BL(0.1)": "BL_k0.1_sample", "BL(0.2)": "BL_k0.2_sample",
    "EqualWeight": "EW", "60/40": "SixtyForty_SPY_IEF",
})
print("reproduction gate: OK")
display_cols = ["ann_return", "ann_vol", "sharpe", "max_dd", "ann_turnover"]
print(summary_table[display_cols].rename(index={"EqualWeight": "EW"}, level=0)
      .to_string(float_format=lambda v: f"{v:.4f}"))

**Reading.** Full-sample Sharpe: 60/40 0.91, EW 0.74, BL(0.1) 0.64, BL(0.2) 0.58; from 2023 the order holds (1.00, 0.91, 0.70, 0.50). BL(0.1) runs at 6.9% volatility against EW's 7.9% and turns over 239% a year against 14%. No fallback or retry at either k; effective N fell below 5 at 16 of 208 rebalances for k = 0.1 and 66 for k = 0.2, in clusters around 2013–14, 2018 and 2023.

## §6 What drove the results

### §6.1 Weights by asset group

Target weights by group at every rebalance (dotted line: split). EW and 60/40 are fixed and appear only in the table.

In [ ]:
fig = ml.plotting.group_stackplot(results, ["BL(0.1)", "BL(0.2)"], figsize=(9, 8),
                                  title_fmt="{name} — target weights by asset group", split_ts=split_ts)
plt.show()

print("Mean target weight by asset group:")
group_means = pd.DataFrame({
    name: ml.diagnostics.weights_by_group(results[name]["wlog"]).mean()
    for name in ["BL(0.1)", "BL(0.2)", "EqualWeight", "60/40"]
}).rename_axis(None).rename(columns={"EqualWeight": "EW"})
print(group_means.to_string(float_format=lambda v: f"{v:.4f}"))

**Reading.** On average BL(0.1) holds 22% equities and 10% fx, against 23% and 8% for EW. The views add most to credit (+4 pp) and take most from real assets (−4 pp).

### §6.2 Concentration and turnover

Effective N, largest weight, UUP weight and trailing-12-rebalance turnover at every rebalance; EW dashed, 60/40 as a line, the effective-N floor of 5 marked.

In [ ]:
c3_names = ["BL(0.1)", "BL(0.2)", "60/40"]
colors = {"BL(0.1)": ml.FAMILY_COLORS[BlackLitterman.family], "BL(0.2)": "#c0392b",
          "60/40": ml.plotting.METHOD_COLORS["60/40"]}
fig = ml.plotting.concentration_panel(
    results, c3_names,
    rows=("effN", "max_w", "highlight", "turnover"), ref="EqualWeight", split_ts=split_ts,
    figsize=(9.5, 14), colors=colors, effn_floor=5.0,
    suptitle="BL(0.1) and BL(0.2) vs 60/40: effN, max weight, UUP weight, turnover",
)
plt.show()

med_effn = {n: float(results[n]["wlog"].apply(ml.diagnostics.effective_n, axis=1).median())
            for n in ["BL(0.1)", "BL(0.2)"]}
print("median effective N: " + "  ".join(f"{n}={v:.2f}" for n, v in med_effn.items()))

**Reading.** Median effective N is 7.60 for BL(0.1) and 6.44 for BL(0.2). Turnover is 239% a year for BL(0.1) and 270% for BL(0.2), against 14% for EW and 11% for 60/40.

### §6.3 What the views did: BL − EW

With k = 0 the model is EW, so BL − EW isolates the views. First each strategy's ex-post CAPM regression (daily simple excess return on SPY − BIL, plain OLS), then the split of the active return into a β part and an α part. Exploratory; notebook 08 tests these gaps.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_strategy_table = ml.inference.capm_table(results, ["BL(0.1)", "BL(0.2)", "EqualWeight", "60/40"], MKT_simple, rf_daily)
print(capm_strategy_table.rename(index={"EqualWeight": "EW"}).to_string(float_format=lambda v: f"{v:.4f}"))

**Reading.** Realized β is 0.29 for BL(0.1), 0.23 for BL(0.2), 0.36 for EW and 0.55 for 60/40. No α is large: t = 0.14 for BL(0.1), 0.32 for BL(0.2), 0.49 for EW and 1.66 for 60/40.

In [ ]:
bl_pairs = [("BL(0.1)", "EqualWeight"), ("BL(0.2)", "EqualWeight")]
bl_paired, bl_paired_series = ml.inference.paired_table(results, bl_pairs, split_ts)
bl_diff_capm = ml.inference.diff_regression(bl_paired_series, bl_pairs, MKT_simple)
decomp = ml.inference.beta_decomposition(capm_strategy_table, bl_paired, bl_diff_capm, bl_pairs,
                                         results, rf_daily, MKT_simple)

for name, ref in bl_pairs:
    r = decomp[f"{name} - {ref}"]
    print(f"=== {name} vs EW (full window) ===")
    print(f"  d_alpha (ann)         = {r['d_alpha']:.4%}")
    print(f"  d_beta                = {r['d_beta']:.4f}")
    print(f"  ann mean MKT excess   = {r['mkt_ann_mean']:.4%}")
    print(f"  beta_part = d_beta * mkt_ann_mean = {r['beta_part']:.4%}")
    print(f"  d_alpha + beta_part   = {r['decomposed']:.4%}")
    print(f"  realized ann active return (full)    = {r['realized_diff_full']:.4%}")
    print(f"  identity check (0 by OLS construction) = {r['identity_check']:.4%}")
    print()

**Reading.** BL(0.1) trails EW by 1.36% a year over the full sample: lower β explains −0.96% (Δβ −0.0655 at a 14.72% market excess return) and α the other −0.40%, which includes about 0.23% of extra trading cost (239% against 14% turnover at 10 bp). At k = 0.2 the gap is 1.97%, of which the β part is −1.80%. Notebook 08 finds BL(0.1)'s Sharpe gap to EW not significant.

Cumulative BL − EW (solid: β-adjusted; dashed: total). Exploratory.

In [ ]:
fig = ml.plotting.cumulative_paired(
    bl_paired_series, bl_pairs, split_ts,
    beta={f"{a} - {b}": bl_diff_capm.loc[f"{a} - {b}", "beta_diff"] for a, b in bl_pairs},
    mkt_excess=MKT_simple,
    colors={"BL(0.1) - EqualWeight": ml.FAMILY_COLORS[BlackLitterman.family],
            "BL(0.2) - EqualWeight": "#c0392b"},
    title="Exploratory: cumulative BL − EW, solid = β-adjusted, dashed = total",
)
plt.show()

**Reading.** The dashed lines end near 17.3 years times the active returns above (about −24% and −34%), the β-adjusted lines near 17.3 years times Δα (about −7% and −3%): most of the gap is β. BL(0.1)'s β-adjusted gap was positive from about 2015 to 2019 and has been negative since 2020.

## §7 TAKEAWAY — momentum views on an EW prior lower β and add turnover, not α

- **No views, no bets.** The EW-implied prior makes k = 0 hold EW (to 3.66e-07); τ cancels, and k sets the size of the momentum tilt (§2).
- **k is a concentration knob.** At 2022-12-31, 11 of 13 signs were negative, one risk-off bet that a fully invested long-only book can hold only in DBC and UUP; effective N falls 7.58 → 3.47 → 1.78 for k = 0.1, 0.2, 0.4.
- **Against the benchmarks:** full-sample Sharpe 0.64 against 0.74 for EW and 0.91 for 60/40, and 0.70 against 0.91 and 1.00 from 2023, with 239% turnover a year against 14% and 11%.
- **The views mostly cut β.** BL(0.1) trails EW by 1.36% a year, −0.96% of it from lower β (about 0.23% of the rest is trading cost); at k = 0.2 the β part is −1.80% of −1.97%. Notebook 08 finds the Sharpe gap not significant.
- **Concentration is episodic.** Effective N fell below 5 at 16 of 208 rebalances (66 at k = 0.2), with no fallbacks or retries.

The risk-off view needs a cash leg this long-only, fully invested book lacks. Notebook 04 keeps the models and changes Σ.